# 🐾 ระบบดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนม (Mammalia Species Data Retrieval)

สมุดบันทึกนี้ออกแบบมาสำหรับใช้งานบน **Google Colab** เพื่อดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนม (Mammals) จากฐานข้อมูลระดับโลก **GBIF (Global Biodiversity Information Facility) API** ซึ่งใช้งานได้ฟรีโดยไม่ต้องมี API Key

---

### 1. ติดตั้งและนำเข้าไลบรารีที่จำเป็น (Import Libraries)

In [3]:
import pandas as pd
import numpy as np
import requests
import os
import json
import shutil
import random
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models, optimizers
# from google.colab import files # สำหรับดาวน์โหลดไฟล์บน Colab


# 1. เชื่อมต่อกับ Google Drive (ยกเลิก Comment เมื่อรันบน Colab จริง)
from google.colab import drive
drive.mount('/content/drive')

print("TensorFlow Version:", tf.__version__)
print("GPU Available:", tf.config.list_physical_devices('GPU'))


Mounted at /content/drive
TensorFlow Version: 2.20.0
GPU Available: []


### 2. กำหนดฟังก์ชันดึงข้อมูลสัตว์เลี้ยงลูกด้วยนม
เราจะดึงข้อมูลผ่าน GBIF Species Search API โดยระบุ `classKey=359` (ซึ่งรหัส 359 คือชั้น **Mammalia** หรือสัตว์เลี้ยงลูกด้วยนม)

In [4]:
def fetch_mammal_species(limit=100, offset=0):
    """
    ฟังก์ชันดึงข้อมูลสายพันธุ์สัตว์เลี้ยงลูกด้วยนมจาก GBIF API
    """
    url = "https://api.gbif.org/v1/species/search"
    params = {
        "classKey": 359,        # 359 = Mammalia (สัตว์เลี้ยงลูกด้วยนม)
        "rank": "SPECIES",      # ดึงเฉพาะระดับระดับชนิด (Species)
        "status": "ACCEPTED",    # ดึงเฉพาะชื่อที่เป็นที่ยอมรับทางอนุกรมวิธาน
        "limit": limit,
        "offset": offset
    }
    
    try:
        response = requests.get(url, params=params)
        response.raise_for_status() # ตรวจสอบ Error
        data = response.json()
        return data.get("results", [])
    except Exception as e:
        print(f"เกิดข้อผิดพลาดในการดึงข้อมูล: {e}")
        return []

### 3. เริ่มดึงข้อมูลและแปลงให้อยู่ในรูปแบบตาราง (Pandas DataFrame)
คุณสามารถปรับเปลี่ยนตัวแปร `total_to_fetch` ด้านล่างเพื่อดึงข้อมูลในจำนวนที่ต้องการได้ (ตัวอย่างนี้ตั้งไว้ที่ 200 รายการ)

In [ ]:
# กำหนดจำนวนที่ต้องการดึง
total_to_fetch = 200
limit_per_request = 100
mammal_list = []

print("กำลังเริ่มต้นดึงข้อมูลจาก GBIF API...")
for offset in range(0, total_to_fetch, limit_per_request):
    print(f"ดึงข้อมูลตำแหน่งที่ {offset} ถึง {offset + limit_per_request}...")
    results = fetch_mammal_species(limit=limit_per_request, offset=offset)
    if not results:
        break
    mammal_list.extend(results)

print(f"ดึงข้อมูลเสร็จสิ้น! ได้รับข้อมูลทั้งหมด {len(mammal_list)} รายการ")

### 4. จัดระเบียบข้อมูลและแสดงผล (Data Processing & Display)
เราจะเลือกเฉพาะคอลัมน์สำคัญ เช่น ชื่อวิทยาศาสตร์, วงศ์ (Family), อันดับ (Order), สกุล (Genus) และนำมาสร้างเป็น DataFrame

In [ ]:
# แปลงข้อมูลเป็น List ของ Dictionary ที่ดูง่าย
processed_data = []
for mammal in mammal_list:
    processed_data.append({
        "Scientific Name": mammal.get("scientificName"),
        "Canonical Name": mammal.get("canonicalName"),
        "Kingdom": mammal.get("kingdom"),
        "Phylum": mammal.get("phylum"),
        "Class": mammal.get("class"),
        "Order": mammal.get("order"),
        "Family": mammal.get("family"),
        "Genus": mammal.get("genus"),
        "Taxonomic Status": mammal.get("taxonomicStatus"),
        "GBIF ID": mammal.get("key")
    })

# สร้าง DataFrame
df = pd.DataFrame(processed_data)

# แสดงตัวอย่างตารางข้อมูล 10 แถวแรก
df.head(10)

### 5. บันทึกข้อมูลและดาวน์โหลดเป็นไฟล์ CSV (Export to CSV)

In [ ]:
# บันทึกเป็นไฟล์ CSV
file_name = "mammal_species_data.csv"
df.to_csv(file_name, index=False, encoding="utf-8-sig")
print(f"บันทึกไฟล์ {file_name} เรียบร้อยแล้ว!")

# ดาวน์โหลดไฟล์ลงเครื่องคอมพิวเตอร์ของคุณ
files.download(file_name)